# CNNで犬猫分類を試そう

この notebook では，CIFAR-10 の猫と犬だけを取り出して，CNN で二値分類する。

MNIST Digits は背景も形もかなり整っている。
それに比べると，犬猫写真は色，背景，向き，大きさがばらばらである。
そのため，同じ CNN でも一気に難しくなる。


## このノートの流れ

まずは CIFAR-10 から猫と犬だけを取り出し，自分で小さな CNN を訓練する。
後半では，ImageNet で訓練済みの ResNet-50 も使う。

自分で訓練するモデルと，すでに大きなデータで学習されたモデルを比べると，画像分類の難しさが見えやすい。


## 準備

CIFAR-10 は `data/cifar-10-batches-py/` に展開済みのものを使う。
`cat` と `dog` の2クラスだけにして，授業中に試しやすいサイズにする。


In [ ]:
from __future__ import annotations

import pickle
import random
import tempfile
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import transformers
from jaxtyping import Float, Int64
from matplotlib import font_manager
from PIL import Image
from sklearn.metrics import accuracy_score
from torch import nn
from torch.utils.data import Dataset
from torchvision import transforms
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    Trainer,
    TrainingArguments,
    pipeline,
)
from transformers.trainer_utils import EvalPrediction

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

DATA_ROOT = Path("data") if Path("data").exists() else Path("../data")

JAPANESE_FONT_CANDIDATES = [
    "Hiragino Sans",
    "Hiragino Maru Gothic Pro",
    "YuGothic",
    "Noto Sans CJK JP",
    "Arial Unicode MS",
]
available_font_names = {font.name for font in font_manager.fontManager.ttflist}
for font_name in JAPANESE_FONT_CANDIDATES:
    if font_name in available_font_names:
        plt.rcParams["font.family"] = font_name
        break
plt.rcParams["axes.unicode_minus"] = False

print(f"device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")


## CIFAR-10 から猫と犬だけを取り出す

CIFAR-10 は 32 x 32 ピクセルの小さなカラー画像である。
ここでは `cat` を0，`dog` を1として扱う。


In [ ]:
CIFAR10_DIR = DATA_ROOT / "cifar-10-batches-py"
CIFAR10_CAT_LABEL = 3
CIFAR10_DOG_LABEL = 5
TRAIN_SAMPLES_PER_LABEL = 1200
EVAL_SAMPLES_PER_LABEL = 300
label_names = ["cat", "dog"]
label_names_ja = ["猫", "犬"]


def read_cifar10_batch(path: Path) -> "tuple[np.ndarray, np.ndarray]":
    """CIFAR-10 の batch ファイルから画像とラベルを読む."""
    with path.open("rb") as file:
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", message=r"dtype\(\): align should be passed")
            batch = pickle.load(file, encoding="latin1")
    data = batch["data"]
    labels = np.asarray(batch["labels"], dtype=np.int64)
    images = data.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
    return images, labels


def load_cifar10_split(paths: list[Path]) -> "tuple[np.ndarray, np.ndarray]":
    """複数 batch を結合し，cat/dog だけに絞る."""
    image_parts = []
    label_parts = []
    for path in paths:
        images, labels = read_cifar10_batch(path)
        mask = np.isin(labels, [CIFAR10_CAT_LABEL, CIFAR10_DOG_LABEL])
        image_parts.append(images[mask])
        label_parts.append(np.where(labels[mask] == CIFAR10_CAT_LABEL, 0, 1))
    return np.concatenate(image_parts), np.concatenate(label_parts)


def sample_balanced(
    images: np.ndarray,
    labels: np.ndarray,
    samples_per_label: int,
) -> tuple[np.ndarray, np.ndarray]:
    """各ラベルから同じ枚数ずつ取り出す."""
    rng = np.random.default_rng(SEED)
    chosen: list[int] = []
    for label_id in [0, 1]:
        label_indices = np.flatnonzero(labels == label_id)
        rng.shuffle(label_indices)
        chosen.extend(label_indices[:samples_per_label].tolist())
    rng.shuffle(chosen)
    return images[chosen], labels[chosen]


if not CIFAR10_DIR.exists():
    raise FileNotFoundError(
        f"{CIFAR10_DIR} が見つかりません。CIFAR-10 を data/ に展開してください。"
    )

train_paths = [CIFAR10_DIR / f"data_batch_{index}" for index in range(1, 6)]
eval_paths = [CIFAR10_DIR / "test_batch"]

train_images_all, train_labels_all = load_cifar10_split(train_paths)
eval_images_all, eval_labels_all = load_cifar10_split(eval_paths)
train_images, train_labels = sample_balanced(train_images_all, train_labels_all, TRAIN_SAMPLES_PER_LABEL)
eval_images, eval_labels = sample_balanced(eval_images_all, eval_labels_all, EVAL_SAMPLES_PER_LABEL)

print("train samples:", len(train_images))
print("eval samples:", len(eval_images))
print("labels:", " / ".join(label_names_ja))


## 写真を見てみる

32 x 32 の小さな写真なので，人間が見ても少し迷うことがある。
ここが MNIST との大きな違いである。


In [ ]:
def show_cifar_examples(images: np.ndarray, labels: np.ndarray, rows: int = 2, cols: int = 8) -> None:
    """CIFAR-10 の猫犬画像例を表示する."""
    fig, axes = plt.subplots(rows, cols, figsize=(10, 3.2))
    for ax, image, label in zip(axes.ravel(), images, labels, strict=False):
        ax.imshow(image)
        ax.set_title(label_names_ja[int(label)])
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_cifar_examples(train_images, train_labels)


## データ拡張を入れる

犬や猫の写真では，左右反転や少しの切り抜きが自然に起きる。
そこで訓練中だけ，画像を少し動かしてからモデルに見せる。


In [ ]:
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.4914, 0.4822, 0.4465), std=(0.2470, 0.2435, 0.2616)),
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.4914, 0.4822, 0.4465), std=(0.2470, 0.2435, 0.2616)),
])


class CifarDogCatDataset(Dataset[tuple[torch.Tensor, int]]):
    """CIFAR-10 の猫犬画像を返す dataset."""

    def __init__(self, images: np.ndarray, labels: np.ndarray, transform: transforms.Compose) -> None:
        self.images = images
        self.labels = labels.astype(np.int64)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.images)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, int]:
        image = Image.fromarray(self.images[index])
        return self.transform(image), int(self.labels[index])


train_dataset_raw = CifarDogCatDataset(train_images, train_labels, train_transform)
eval_dataset_raw = CifarDogCatDataset(eval_images, eval_labels, eval_transform)


## `Trainer` に渡す形へそろえる

ここも MNIST と同じで，画像は `pixel_values`，正解ラベルは `labels` という名前にする。


In [ ]:
class ImageDatasetForTrainer(Dataset[dict[str, torch.Tensor]]):
    """画像分類 dataset を Trainer 用の辞書形式で返す dataset."""

    def __init__(self, dataset: Dataset) -> None:
        self.dataset = dataset

    def __len__(self) -> int:
        return len(self.dataset)

    def __getitem__(self, index: int) -> dict[str, torch.Tensor]:
        image, label = self.dataset[index]
        return {
            "pixel_values": image,
            "labels": torch.tensor(label, dtype=torch.long),
        }

train_dataset = ImageDatasetForTrainer(train_dataset_raw)
eval_dataset = ImageDatasetForTrainer(eval_dataset_raw)


## 犬猫用の CNN を作る

犬猫画像はカラーなので，入力のチャンネル数は3である。
画像は 32 x 32 なので，pooling を3回通すと 4 x 4 まで小さくなる。


In [ ]:
def make_dog_cat_cnn(num_labels: int) -> nn.Sequential:
    """CIFAR-10 の犬猫用 CNN を1つの Sequential として作る."""
    return nn.Sequential(
        nn.Conv2d(3, 32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Conv2d(64, 128, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Flatten(),
        nn.Linear(128 * 4 * 4, 128),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(128, num_labels),
    )


class ImageClassifierForTrainer(nn.Module):
    """`Trainer` に合わせて `loss` と `logits` を返す接続用クラス."""

    def __init__(self, network: nn.Sequential) -> None:
        super().__init__()
        self.network = network
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(
        self,
        pixel_values: Float[torch.Tensor, "batch 3 32 32"],
        labels: Int64[torch.Tensor, "batch"] | None = None,
    ) -> dict[str, torch.Tensor]:
        logits = self.network(pixel_values)
        output = {"logits": logits}
        if labels is not None:
            output["loss"] = self.loss_fn(logits, labels)
        return output


network = make_dog_cat_cnn(num_labels=len(label_names))
model = ImageClassifierForTrainer(network)
network


## 途中の形だけ見てみる

3色の画像が，畳み込みと pooling でどのように小さくなるかを見る。


In [ ]:
shape_check = torch.zeros(1, 3, 32, 32)
for layer in network:
    shape_check = layer(shape_check)
    print(f"{layer.__class__.__name__:12s} -> {tuple(shape_check.shape)}")


## `Trainer` で訓練する

犬猫は MNIST より難しいので，accuracy がすぐに高くならなくても自然である。
まずは学習が進むかどうかを見る。


In [ ]:
def compute_accuracy(eval_pred: EvalPrediction) -> dict[str, float]:
    """Trainer の予測結果から accuracy を計算する."""
    logits = eval_pred.predictions
    labels = eval_pred.label_ids
    predictions = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(labels, predictions)}

training_args = TrainingArguments(
    output_dir=tempfile.mkdtemp(prefix="cifar-dog-cat-cnn-"),
    num_train_epochs=5,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=256,
    learning_rate=1e-3,
    optim="adamw_torch",
    eval_strategy="epoch",
    save_strategy="no",
    logging_strategy="epoch",
    report_to="none",
    remove_unused_columns=False,
    seed=SEED,
    use_cpu=DEVICE.type == "cpu",
    dataloader_pin_memory=DEVICE.type == "cuda",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_accuracy,
)

train_output = trainer.train()
eval_metrics = trainer.evaluate(eval_dataset)
eval_metrics


## 予測例を見る

間違いがあったら，背景や姿勢に引っ張られていないかを見てみよう。
犬猫写真では，画像そのものを観察することがとても大事である。


In [ ]:
prediction_output = trainer.predict(eval_dataset)
y_pred = np.argmax(prediction_output.predictions, axis=1)
y_true = prediction_output.label_ids
accuracy = accuracy_score(y_true, y_pred)
print(f"eval accuracy: {accuracy:.3f} ({int((y_pred == y_true).sum())}/{len(y_true)})")

fig, axes = plt.subplots(2, 8, figsize=(10, 3.2))
for ax, index in zip(axes.ravel(), range(16), strict=False):
    image = eval_images[index]
    true_label = int(y_true[index])
    predicted_label = int(y_pred[index])
    title_color = "black" if predicted_label == true_label else "crimson"
    ax.imshow(image)
    ax.set_title(
        f"正: {label_names_ja[true_label]}\n予: {label_names_ja[predicted_label]}",
        color=title_color,
        fontsize=10,
    )
    ax.axis("off")
plt.tight_layout()
plt.show()


## 学習済み ResNet-50 も使ってみる

ImageNet で訓練された ResNet-50 は，犬や猫のような自然画像に向いている。
ただし，これは「犬か猫か」の2択モデルではなく，ImageNet の1000種類から近いラベルを出すモデルである。

そのため，犬なら犬種名，猫なら `tabby cat` のようなラベルが上位に出るかを見る。


In [ ]:
HF_IMAGE_MODEL_ID = "microsoft/resnet-50"

sample_indices = []
for label_id in [0, 1]:
    sample_indices.append(int(np.flatnonzero(eval_labels == label_id)[0]))

sample_images = {
    label_names_ja[int(eval_labels[index])]: Image.fromarray(eval_images[index])
    for index in sample_indices
}

for label, image in sample_images.items():
    print(label)
    display(image.resize((160, 160)))


### いちばん簡単な使い方: `pipeline`

`pipeline` を使うと，モデルの読み込みから予測までを短いコードで試せる。
初回は Hugging Face Hub からモデルを取得するため，ネットワーク接続が必要である。


In [ ]:
try:
    image_classifier = pipeline(
        task="image-classification",
        model=HF_IMAGE_MODEL_ID,
        device=0 if DEVICE.type == "cuda" else -1,
    )
    for label, image in sample_images.items():
        print()
        print(f"入力: {label}")
        predictions = image_classifier(image, top_k=5)
        for rank, item in enumerate(predictions, start=1):
            print(f"{rank}. {item['label']}: {item['score']:.3f}")
except Exception as exc:
    print("事前学習済みモデルを読み込めませんでした。")
    print("ネットワークや Hugging Face Hub への接続を確認してください。")
    print(f"{type(exc).__name__}: {exc}")


### 中身も見たい使い方: `AutoImageProcessor` と `AutoModelForImageClassification`

`AutoImageProcessor` は，画像をそのモデルが学習したときと同じ形に整える係である。
`AutoModelForImageClassification` は，画像から各クラスの点数を出すモデル本体である。


In [ ]:
try:
    image_processor = AutoImageProcessor.from_pretrained(HF_IMAGE_MODEL_ID)
    hf_image_model = AutoModelForImageClassification.from_pretrained(HF_IMAGE_MODEL_ID).to(DEVICE)

    first_label, first_image = next(iter(sample_images.items()))
    inputs = image_processor(images=first_image, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = hf_image_model(**inputs)

    probabilities = outputs.logits.softmax(dim=-1)[0].cpu()
    top_scores, top_indices = probabilities.topk(5)
    print(f"入力: {first_label}")
    for rank, (score, index) in enumerate(zip(top_scores, top_indices, strict=True), start=1):
        label = hf_image_model.config.id2label[int(index)]
        print(f"{rank}. {label}: {float(score):.3f}")
except Exception as exc:
    print("事前学習済みモデルを読み込めませんでした。")
    print("この節はネットワーク接続があるときにもう一度試してください。")
    print(f"{type(exc).__name__}: {exc}")


## まとめ

この notebook では，CIFAR-10 の猫と犬を使って，MNIST より難しい画像分類を試した。

- 犬猫写真は，色，背景，向き，大きさがばらばらである。
- カラー画像なので，入力チャンネルは3になる。
- データ拡張では，切り抜きや左右反転を使った。
- 小さな CNN を自分で訓練すると，写真分類の難しさが見える。
- ImageNet で学習済みの ResNet-50 は，自然画像に向いた強いモデルとして使える。
